# FaceForensics++ C23 Preprocessing

Converts FF++ C23 videos into `.npz` files compatible with FakeAVCeleb and LAV-DF.

### Output format (identical to other datasets)
- `frames`: `(16, 224, 224, 3)` uint8 — 16 uniformly sampled frames
- `audio`: `(16, 128, 32)` float32 — mel spectrogram segments
- `label`: scalar int
  - `0` = Real/Real (original videos)
  - `1` = FakeVideo/RealAudio (all FF++ manipulations)

### Sampling strategy
- Uses `FF++_Metadata_Shuffled.csv` as the file list (all 7000 entries)
- **250 samples per manipulation type × 6 = 1500 fake (label 1)**
- **500 real (label 0)**
- 80/20 random train/test split per manipulation type
- Weighted sampling during joint training handles class imbalance


In [1]:
# ── CELL 1: SETUP ─────────────────────────────────────────────────────────
!pip install librosa opencv-python-headless tqdm kagglehub retina-face -q
!apt-get install -y ffmpeg -q
print('Setup done.')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Setup done.


In [14]:
# ── CELL 2: IMPORTS & DRIVE ───────────────────────────────────────────────
import os, random, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import librosa
from tqdm import tqdm
from google.colab import drive
from retinaface import RetinaFace

drive.mount('/content/drive')

random.seed(42)
np.random.seed(42)
print('Imports done.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports done.


In [3]:
# ── CELL 3: DOWNLOAD DATASET ──────────────────────────────────────────────
import kagglehub

path    = kagglehub.dataset_download('xdxd003/ff-c23')
FF_ROOT = os.path.join(path, 'FaceForensics++_C23')

print('FF_ROOT:', FF_ROOT)
print('Contents:', sorted(os.listdir(FF_ROOT)))

100%|██████████| 16.7G/16.7G [02:25<00:00, 123MB/s] 

Extracting files...


FF_ROOT: /root/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23
Contents: ['DeepFakeDetection', 'Deepfakes', 'Face2Face', 'FaceShifter', 'FaceSwap', 'NeuralTextures', 'csv', 'original']


In [4]:
# ── CELL 4: CONFIG ────────────────────────────────────────────────────────
OUT_TRAIN = '/content/drive/MyDrive/Deepfake/ff_processed/train'
OUT_TEST  = '/content/drive/MyDrive/Deepfake/ff_processed/test'
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST,  exist_ok=True)

# Preprocessing params — identical to FakeAVCeleb and LAV-DF
FRAME_SIZE    = 224
NUM_FRAMES    = 16
NUM_AUDIO_SEG = 16
SR            = 16000
N_MELS        = 128
TARGET_MEL_T  = 32

SAMPLES_PER_TYPE = 100   # × 6 types = 1500 fake total
REAL_SAMPLES     = 200
TRAIN_RATIO      = 0.8

LABEL_REAL = 0  # Real/Real
LABEL_FAKE = 1  # FakeVideo/RealAudio

FAKE_TYPES = [
    'DeepFakeDetection',
    'Deepfakes',
    'Face2Face',
    'FaceShifter',
    'FaceSwap',
    'NeuralTextures',
]

print(f'Plan: {len(FAKE_TYPES)} types × {SAMPLES_PER_TYPE} = {len(FAKE_TYPES)*SAMPLES_PER_TYPE} fake + {REAL_SAMPLES} real')

Plan: 6 types × 100 = 600 fake + 200 real


In [5]:
# ── CELL 5: LOAD CSV & BUILD ENTRY LIST ───────────────────────────────────
# FF++_Metadata_Shuffled.csv has all 7000 entries (6000 fake + 1000 real)

csv_path = os.path.join(FF_ROOT, 'csv', 'FF++_Metadata_Shuffled.csv')
meta_df  = pd.read_csv(csv_path)

print(f'Total CSV entries: {len(meta_df)}')
print(meta_df['Label'].value_counts())

all_entries = []  # (video_path, label, manipulation_type)

# Real videos
real_rows = meta_df[meta_df['Label'] == 'REAL'].head(REAL_SAMPLES)
for _, row in real_rows.iterrows():
    vpath = os.path.join(FF_ROOT, row['File Path'])
    all_entries.append((vpath, LABEL_REAL, 'original'))
print(f'\nReal selected: {len(real_rows)}')

# Fake videos — per manipulation type
for fake_type in FAKE_TYPES:
    rows = meta_df[meta_df['File Path'].str.startswith(fake_type)].head(SAMPLES_PER_TYPE)
    for _, row in rows.iterrows():
        vpath = os.path.join(FF_ROOT, row['File Path'])
        all_entries.append((vpath, LABEL_FAKE, fake_type))
    print(f'{fake_type}: {len(rows)} selected')

print(f'\nTotal: {len(all_entries)}  label dist: {Counter(l for _, l, _ in all_entries)}')

Total CSV entries: 7000
Label
FAKE    6000
REAL    1000
Name: count, dtype: int64

Real selected: 200
DeepFakeDetection: 100 selected
Deepfakes: 100 selected
Face2Face: 100 selected
FaceShifter: 100 selected
FaceSwap: 100 selected
NeuralTextures: 100 selected

Total: 800  label dist: Counter({1: 600, 0: 200})


In [6]:
# ── CELL 6: TRAIN/TEST SPLIT (per manipulation type) ──────────────────────
by_type = defaultdict(list)
for entry in all_entries:
    by_type[entry[2]].append(entry)

train_entries = []
test_entries  = []

for mtype, entries in by_type.items():
    random.shuffle(entries)
    n_train = int(len(entries) * TRAIN_RATIO)
    train_entries.extend(entries[:n_train])
    test_entries.extend(entries[n_train:])

# Flatten to (path, label)
train_entries = [(v, l) for v, l, _ in train_entries]
test_entries  = [(v, l) for v, l, _ in test_entries]
random.shuffle(train_entries)
random.shuffle(test_entries)

print(f'Train: {len(train_entries)}  label dist: {Counter(l for _, l in train_entries)}')
print(f'Test : {len(test_entries)}   label dist: {Counter(l for _, l in test_entries)}')

Train: 640  label dist: Counter({1: 480, 0: 160})
Test : 160   label dist: Counter({1: 120, 0: 40})


In [7]:
# ── CELL 7: EXTRACTION FUNCTIONS (with RetinaFace face detection) ─────────

def detect_face_box(frame):
    try:
        faces = RetinaFace.detect_faces(frame)
        if isinstance(faces, dict):
            face = list(faces.values())[0]
            return face['facial_area']
    except:
        pass
    return None


def crop_face(frame, face_box):
    """Crop face region or fall back to center crop if no face detected."""
    h, w, _ = frame.shape
    if face_box is not None:
        x1, y1, x2, y2 = face_box
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        crop = frame[y1:y2, x1:x2]
    else:
        size = min(h, w) // 2
        cx, cy = w // 2, h // 2
        crop = frame[cy-size:cy+size, cx-size:cx+size]
    return cv2.resize(
        cv2.cvtColor(crop, cv2.COLOR_BGR2RGB),
        (FRAME_SIZE, FRAME_SIZE)
    )


def detect_face_box_from_video(video_path, max_frames=30):
    """Detect face box from the first max_frames frames of a video."""
    cap = cv2.VideoCapture(video_path)
    face_box = None
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        face_box = detect_face_box(frame)
        if face_box is not None:
            break
    cap.release()
    return face_box


def extract_frames_uniform(video_path, n=NUM_FRAMES):
    face_box = detect_face_box_from_video(video_path)
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return []
    indices = np.linspace(0, total - 1, n, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        frames.append(crop_face(frame, face_box))
    cap.release()
    return frames


def load_audio(video_path):
    """Extract audio. Returns silence if no audio track found."""
    tmp = '/tmp/ff_audio.wav'
    subprocess.run(
        ['ffmpeg', '-y', '-i', video_path, '-ac', '1', '-ar', str(SR), tmp],
        capture_output=True
    )
    if os.path.exists(tmp) and os.path.getsize(tmp) > 1000:
        try:
            return librosa.load(tmp, sr=SR)
        except Exception:
            pass
    # No audio track — return silence
    cap      = cv2.VideoCapture(video_path)
    fps      = cap.get(cv2.CAP_PROP_FPS) or 25
    total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    duration = total / fps
    return np.zeros(int(SR * duration), dtype=np.float32), SR


def mel_segment(segment, sr):
    mel = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=N_MELS)
    mel = librosa.power_to_db(mel)
    if mel.shape[1] < TARGET_MEL_T:
        mel = np.pad(mel, ((0, 0), (0, TARGET_MEL_T - mel.shape[1])))
    else:
        mel = mel[:, :TARGET_MEL_T]
    return mel


def extract_audio_uniform(audio, sr, n=NUM_AUDIO_SEG):
    if len(audio) < n:
        return []
    seg_len = max(1, len(audio) // n)
    return [mel_segment(audio[i * seg_len:(i + 1) * seg_len], sr) for i in range(n)]


print('Extraction functions ready (with RetinaFace face detection).')

Extraction functions ready (with RetinaFace face detection).


In [8]:
# ── CELL 8: SANITY CHECK ──────────────────────────────────────────────────

def process_video(video_path, label):
    try:
        frames = extract_frames_uniform(video_path)
        if len(frames) < 8:
            return None
        while len(frames) < NUM_FRAMES:
            frames.append(frames[-1])
        frames = frames[:NUM_FRAMES]

        audio_raw, sr = load_audio(video_path)
        audio_feats   = extract_audio_uniform(audio_raw, sr)
        if len(audio_feats) < 8:
            return None
        while len(audio_feats) < NUM_AUDIO_SEG:
            audio_feats.append(audio_feats[-1])
        audio_feats = audio_feats[:NUM_AUDIO_SEG]

        return (
            np.array(frames,      dtype=np.uint8),
            np.array(audio_feats, dtype=np.float32),
            label
        )
    except Exception:
        return None


real_sample = next(v for v, l in train_entries if l == LABEL_REAL)
fake_sample = next(v for v, l in train_entries if l == LABEL_FAKE)

print('Testing on one real video...')
r = process_video(real_sample, LABEL_REAL)
print(f'  frames={r[0].shape}  audio={r[1].shape}  label={r[2]}' if r else '  FAILED')

print('Testing on one fake video...')
f = process_video(fake_sample, LABEL_FAKE)
print(f'  frames={f[0].shape}  audio={f[1].shape}  label={f[2]}' if f else '  FAILED')

Testing on one real video...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5


26-03-09 09:06:26 - Directory /root/.deepface created
26-03-09 09:06:26 - Directory /root/.deepface/weights created
26-03-09 09:06:26 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


100%|██████████| 119M/119M [00:00<00:00, 416MB/s] 


  frames=(16, 224, 224, 3)  audio=(16, 128, 32)  label=0
Testing on one fake video...
  frames=(16, 224, 224, 3)  audio=(16, 128, 32)  label=1


In [9]:
# ── CELL 9: PREPROCESSING LOOP ────────────────────────────────────────────

def run_preprocessing(entries, out_dir, prefix='ff'):
    done      = set(f for f in os.listdir(out_dir) if f.endswith('.npz'))
    total_new = 0
    print(f'Already saved: {len(done)} files')

    for video_path, label in tqdm(entries, desc=f'Processing {prefix}'):
        vid_id   = os.path.splitext(os.path.basename(video_path))[0]
        parent   = os.path.basename(os.path.dirname(video_path))
        out_name = f'{prefix}_{label}_{parent}_{vid_id}.npz'

        if out_name in done:
            continue

        result = process_video(video_path, label)
        if result is None:
            continue

        frames, audio, lbl = result
        np.savez_compressed(
            os.path.join(out_dir, out_name),
            frames=frames,
            audio=audio,
            label=np.array(lbl)
        )
        done.add(out_name)
        total_new += 1

    print(f'New files saved : {total_new}')
    print(f'Total in dir    : {len(os.listdir(out_dir))}')

print('Loop ready.')

Loop ready.


In [10]:
# ── CELL 10: PREPROCESS TRAIN ─────────────────────────────────────────────
print('=== TRAIN ===')
run_preprocessing(train_entries, OUT_TRAIN, prefix='ff')

=== TRAIN ===
Already saved: 0 files


Processing ff: 100%|██████████| 640/640 [3:11:06<00:00, 17.92s/it]

New files saved : 640
Total in dir    : 640


In [11]:
# ── CELL 11: PREPROCESS TEST ──────────────────────────────────────────────
print('=== TEST ===')
run_preprocessing(test_entries, OUT_TEST, prefix='ff')

=== TEST ===
Already saved: 0 files


Processing ff: 100%|██████████| 160/160 [55:07<00:00, 20.67s/it]

New files saved : 160
Total in dir    : 160


In [12]:
# ── CELL 12: VERIFY ───────────────────────────────────────────────────────
print('=== VERIFICATION ===')
for split_name, out_dir in [('TRAIN', OUT_TRAIN), ('TEST', OUT_TEST)]:
    files  = [f for f in os.listdir(out_dir) if f.endswith('.npz')]
    labels = [int(f.split('_')[1]) for f in files]
    print(f'\n{split_name}: {len(files)} total')
    for c, cnt in sorted(Counter(labels).items()):
        name = 'Real/Real' if c == 0 else 'FakeVideo/RealAudio'
        print(f'  Label {c} ({name}): {cnt}')

sample_f = os.path.join(OUT_TRAIN, [f for f in os.listdir(OUT_TRAIN) if f.endswith('.npz')][0])
d = np.load(sample_f)
print(f'\nSample: {os.path.basename(sample_f)}')
print(f'  frames : {d["frames"].shape}  (expect (16,224,224,3))')
print(f'  audio  : {d["audio"].shape}   (expect (16,128,32))')
print(f'  label  : {d["label"]}')

=== VERIFICATION ===

TRAIN: 640 total
  Label 0 (Real/Real): 160
  Label 1 (FakeVideo/RealAudio): 480

TEST: 160 total
  Label 0 (Real/Real): 40
  Label 1 (FakeVideo/RealAudio): 120

Sample: ff_1_FaceShifter_802_885.npz
  frames : (16, 224, 224, 3)  (expect (16,224,224,3))
  audio  : (16, 128, 32)   (expect (16,128,32))
  label  : 1


In [16]:
import numpy as np
import os

datasets = {
    'FakeAVCeleb': '/content/drive/MyDrive/Deepfake/processed_dataset_v7',
    'LAV-DF'     : '/content/drive/MyDrive/Deepfake/lavdf_ts_processed/train',
    'FF++'       : '/content/drive/MyDrive/Deepfake/ff_processed/train',
}

for name, path in datasets.items():
    files = [f for f in os.listdir(path) if f.endswith('.npz')]
    sample = np.load(os.path.join(path, files[0]))
    print(f'\n{name}')
    print(f'  Total files : {len(files)}')
    print(f'  frames shape: {sample["frames"].shape}')
    print(f'  audio shape : {sample["audio"].shape}')
    print(f'  label       : {sample["label"]}')
    print(f'  frames dtype: {sample["frames"].dtype}')
    print(f'  audio dtype : {sample["audio"].dtype}')
    print(f'  frames range: {sample["frames"].min()} - {sample["frames"].max()}')
    print(f'  audio range : {sample["audio"].min():.2f} - {sample["audio"].max():.2f}')


FakeAVCeleb
  Total files : 20559
  frames shape: (16, 224, 224, 3)
  audio shape : (16, 128, 32)
  label       : 3
  frames dtype: uint8
  audio dtype : float32
  frames range: 0 - 255
  audio range : -86.98 - 17.74

LAV-DF
  Total files : 800
  frames shape: (16, 224, 224, 3)
  audio shape : (16, 128, 32)
  label       : 0
  frames dtype: uint8
  audio dtype : float32
  frames range: 0 - 255
  audio range : -65.46 - 24.56

FF++
  Total files : 640
  frames shape: (16, 224, 224, 3)
  audio shape : (16, 128, 32)
  label       : 1
  frames dtype: uint8
  audio dtype : float32
  frames range: 0 - 254
  audio range : -100.00 - 0.00


In [17]:
print(OUT_TRAIN)  # run in LAV-DF notebook
print(OUT_TRAIN)  # run in FF++ notebook

/content/drive/MyDrive/Deepfake/ff_processed/train
/content/drive/MyDrive/Deepfake/ff_processed/train


In [18]:
print(OUT_TRAIN)

/content/drive/MyDrive/Deepfake/ff_processed/train
